<a href="https://colab.research.google.com/github/radhikatyagi388/Ai_60_Day_Challange/blob/main/Document_Retrieval_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Day 11 — Build Your First Document Retrieval System

## Objective

Extend the modular NLP pipeline from Day 10 into a document retrieval system.

The system will:

- Store a knowledge base of 20+ documents
- Preprocess documents using the Day 10 PreprocessingModule
- Convert documents into TF-IDF vectors
- Accept a user query
- Calculate cosine similarity against every document
- Return the top K most relevant documents
- Reject results when the highest similarity score is below 0.1
- Analyze retrieval failures
- Explore vocabulary mismatch and the limitations of TF-IDF

In [1]:
pip install scikit-learn numpy

In [2]:
import re
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# 1. DAY 10 PIPELINE
# ============================================================

class Pipeline:

    def __init__(self):
        self.vectorizer = TfidfVectorizer(
            stop_words="english"
        )

    def preprocess(self, text):
        """
        Basic text preprocessing:
        - Convert to lowercase
        - Remove special characters
        - Remove extra spaces
        """
        text = text.lower()
        text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()

        return text

    def preprocess_documents(self, documents):
        return [
            self.preprocess(document)
            for document in documents
        ]

    def fit_transform(self, documents):
        processed_documents = self.preprocess_documents(documents)

        matrix = self.vectorizer.fit_transform(
            processed_documents
        )

        return matrix

    def transform(self, documents):
        processed_documents = self.preprocess_documents(documents)

        return self.vectorizer.transform(
            processed_documents
        )


# ============================================================
# 2. KNOWLEDGE BASE
# ============================================================

documents = [

    "Python is a high-level programming language used for web development, automation, data science, and scripting.",

    "Java is an object-oriented programming language widely used for enterprise software, Android applications, and backend systems.",

    "C++ is a compiled programming language known for performance, systems programming, game development, and competitive programming.",

    "JavaScript is a programming language used to create interactive web pages and is commonly used with HTML and CSS.",

    "HTML provides the structure of web pages using elements such as headings, paragraphs, links, tables, and forms.",

    "CSS controls the presentation of web pages, including colors, spacing, layout, fonts, and responsive design.",

    "SQL is a language for querying and managing data stored in relational databases such as MySQL and PostgreSQL.",

    "MongoDB is a document-oriented NoSQL database that stores data in flexible JSON-like documents.",

    "Git is a version control system that tracks changes in source code and helps developers collaborate.",

    "GitHub is a platform for hosting Git repositories, reviewing code, managing issues, and collaborating on software projects.",

    "HTTP is an application-layer protocol used for communication between web browsers and web servers.",

    "DNS translates domain names such as example.com into IP addresses used by computers on networks.",

    "TCP is a reliable transport protocol that provides ordered delivery, error checking, and retransmission of data.",

    "UDP is a connectionless transport protocol that sends datagrams without guaranteeing delivery or ordering.",

    "An operating system manages hardware resources and provides services for applications, including process and memory management.",

    "Cloud computing provides on-demand access to computing resources such as servers, storage, and databases over a network.",

    "Cybersecurity protects computers, networks, applications, and data from attacks such as malware, phishing, and unauthorized access.",

    "Machine learning enables computers to learn patterns from data and make predictions or decisions without explicit rules for every case.",

    "Information retrieval finds relevant documents from a collection in response to a user's query.",

    "TF-IDF represents text using term importance based on word frequency in a document and rarity across the collection."
]


# ============================================================
# 3. CREATE PIPELINE AND TF-IDF MATRIX
# ============================================================

pipeline = Pipeline()

corpus_matrix = pipeline.fit_transform(documents)

print("Knowledge base size:", len(documents))
print("TF-IDF matrix shape:", corpus_matrix.shape)


# ============================================================
# 4. RETRIEVAL FUNCTION
# ============================================================

def retrieve(query, corpus_matrix, top_k=3):

    # Convert query into TF-IDF vector
    query_vector = pipeline.transform([query])

    # Calculate cosine similarity
    similarities = cosine_similarity(
        query_vector,
        corpus_matrix
    )[0]

    # Sort document indices by similarity
    ranked_indices = np.argsort(
        similarities
    )[::-1]

    # Highest similarity
    highest_score = similarities[
        ranked_indices[0]
    ]

    # Relevance threshold
    if highest_score < 0.1:
        return "No relevant document found"

    results = []

    for index in ranked_indices[:top_k]:

        results.append({
            "document_id": index + 1,
            "document": documents[index],
            "score": round(
                float(similarities[index]),
                3
            )
        })

    return results


# ============================================================
# 5. TEST QUERIES
# ============================================================

test_queries = [

    # Normal successful queries
    "What language is used for Android applications?",

    "Which protocol provides reliable data delivery?",

    "How can I translate a domain name into an IP address?",

    "Which system tracks source code changes?",

    "What technology allows computers to learn from data?",

    "Which technology protects computers and networks from attacks?",

    # Ambiguous queries
    "programming language",

    "network protocol",

    # Completely out-of-domain
    "recipe for chocolate cake",

    "best tourist places in Delhi"
]


# ============================================================
# 6. RUN TESTS
# ============================================================

print("\n" + "=" * 70)
print("RETRIEVAL TEST RESULTS")
print("=" * 70)

for i, query in enumerate(test_queries, 1):

    print(f"\nQuery {i}: {query}")

    results = retrieve(
        query,
        corpus_matrix,
        top_k=3
    )

    if isinstance(results, str):

        print(results)

    else:

        for result in results:

            print(
                f"Document {result['document_id']} "
                f"| Score: {result['score']}"
            )

            print(
                f"  {result['document']}"
            )


# ============================================================
# 7. FAILURE ANALYSIS
# ============================================================

print("\n" + "=" * 70)
print("FAILURE ANALYSIS")
print("=" * 70)

failure_analysis = {

    "programming language":
        "The query is ambiguous because several documents describe different programming languages, so TF-IDF cannot determine which language the user actually wants.",

    "network protocol":
        "The query is ambiguous because multiple documents describe network protocols such as TCP and UDP, so there is no single uniquely correct document.",

    "recipe for chocolate cake":
        "The query is outside the knowledge base, so no document contains meaningful vocabulary related to recipes or chocolate cake.",

    "best tourist places in Delhi":
        "The query is outside the knowledge base because the corpus contains computer science documents rather than travel information."
}

for query, diagnosis in failure_analysis.items():

    print(f"\nQuery: {query}")
    print("Diagnosis:", diagnosis)

Knowledge base size: 20
TF-IDF matrix shape: (20, 167)

RETRIEVAL TEST RESULTS

Query 1: What language is used for Android applications?
Document 2 | Score: 0.494
  Java is an object-oriented programming language widely used for enterprise software, Android applications, and backend systems.
Document 4 | Score: 0.278
  JavaScript is a programming language used to create interactive web pages and is commonly used with HTML and CSS.
Document 1 | Score: 0.189
  Python is a high-level programming language used for web development, automation, data science, and scripting.

Query 2: Which protocol provides reliable data delivery?
Document 13 | Score: 0.604
  TCP is a reliable transport protocol that provides ordered delivery, error checking, and retransmission of data.
Document 14 | Score: 0.269
  UDP is a connectionless transport protocol that sends datagrams without guaranteeing delivery or ordering.
Document 11 | Score: 0.117
  HTTP is an application-layer protocol used for communication 

In [3]:
print("\n" + "=" * 70)
print("SYNONYM TEST")
print("=" * 70)

result = retrieve(
    "web page styling",
    corpus_matrix,
    top_k=3
)

print(result)


SYNONYM TEST
[{'document_id': np.int64(11), 'document': 'HTTP is an application-layer protocol used for communication between web browsers and web servers.', 'score': 0.457}, {'document_id': np.int64(1), 'document': 'Python is a high-level programming language used for web development, automation, data science, and scripting.', 'score': 0.223}, {'document_id': np.int64(4), 'document': 'JavaScript is a programming language used to create interactive web pages and is commonly used with HTML and CSS.', 'score': 0.219}]


In [ ]:
## Failure Analysis

The retrieval engine performs well when the query contains words that directly overlap with the vocabulary present in the knowledge base. However, it can produce incorrect results when the query is ambiguous, contains synonyms, or belongs to a domain that is not represented in the corpus.

### 1. Ambiguous query: "programming language"

This query is ambiguous because the knowledge base contains separate documents about Python, Java, C++, and JavaScript. TF-IDF mainly measures lexical overlap and therefore cannot determine which programming language the user intended.

### 2. Ambiguous query: "network protocol"

This query can refer to TCP, UDP, or HTTP. Since several documents contain the words "network" and "protocol", TF-IDF may rank a document that shares common vocabulary even when it is not the user's intended protocol.

### 3. Out-of-domain query: "recipe for chocolate cake"

The knowledge base contains computer science documents and has no meaningful vocabulary related to recipes or chocolate cake. The highest similarity therefore falls below the 0.1 threshold, causing the system to correctly return "No relevant document found."

### 4. Out-of-domain query: "best tourist places in Delhi"

The query is unrelated to the computer science knowledge base. Because there is insufficient vocabulary overlap with the corpus, the similarity score remains below the relevance threshold and the system rejects the query.

### 5. Vocabulary mismatch / synonym failure

TF-IDF depends heavily on exact words shared between the query and documents. For example, a user might ask "How do I style a webpage?" while the document discusses CSS using terms such as "presentation", "colors", "spacing", "layout", and "fonts". Even though the concepts are related, the words may not overlap sufficiently. Consequently, TF-IDF can assign a low similarity score or retrieve an unrelated document.

This limitation occurs because TF-IDF represents text using individual words rather than their underlying meaning. Words such as "car" and "automobile" are treated as different terms even though they have similar meanings.

### Conclusion

The experiment shows that TF-IDF is useful for simple information retrieval when the query and document share important vocabulary. However, it struggles with synonyms, paraphrases, ambiguous queries, and semantic relationships. This motivates the use of embedding-based retrieval, where text is represented as dense vectors that capture semantic meaning. Embeddings can identify that words such as "car" and "automobile" are related even when they are not exactly the same word.


In [ ]:
## Relevance Threshold

A relevance threshold of 0.1 was implemented in the retrieval engine.

After calculating cosine similarity between the query and every document, the system finds the highest similarity score:

```python
highest_score = similarities[ranked_indices[0]]
```

If the highest score is below 0.1:

```python
if highest_score < 0.1:
    return "No relevant document found"
```

This prevents the retrieval engine from returning an apparently relevant document when all available documents have very weak similarity with the query.

For example, an out-of-domain query such as "recipe for chocolate cake" may have a highest similarity score of 0.0. Returning one of the computer science documents would be misleading. Instead, the threshold allows the system to honestly report that no relevant document was found.

The threshold therefore acts as a basic confidence mechanism for the retrieval engine.
